# 6 — Final Forecast Benchmarking, Regression Reports & Visualizations

## Objective

Compare the six required forecasting results on exactly the same held-out weekly targets:

1. SGDRegressor — squared error
2. SGDRegressor — Huber
3. SGDRegressor — epsilon-insensitive
4. MLPRegressor
5. Incremental Mean Ensemble (the four pretrained Stream I predictions)
6. DCRNN (state-level forecasts summed to the national count)

The principal figure overlays actual national weekly accident counts (blue) and all six model forecasts (different colours). Additional figures inspect fitted values, residuals, error distributions and time-varying accuracy.

### Metrics

- MAE
- RMSE
- Adjusted R² (reported with a feature-count caveat)
- WAPE
- Mean Poisson Deviance
- sMAPE

`log_loss` is not used: it is defined for classification probability distributions, not continuous count forecasts. Poisson deviance is the relevant log-likelihood/deviance-style metric for non-negative count targets.

### Incomplete boundary weeks

Notebook 2 retains every valid source report, including reports in the first/last observed calendar week. The exposure-day count is stored as metadata. The primary comparable metrics and plots use complete seven-day target weeks only; partial boundary weeks remain in the saved prediction files and are reported separately rather than silently discarded from the source-derived count table.

## 0. Paths, metrics and model prediction inputs

In [ ]:
from pathlib import Path
import os, json, warnings, gc, time, copy, random, sys
import numpy as np
import pandas as pd

SEED = 34
random.seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for _p in (PROJECT_ROOT, ARTIFACT_DIR, MODEL_DIR, REPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# Set this only when more than one prior notebook output is attached as a Kaggle input.
ARTIFACT_INPUT_ROOT = None  # e.g. Path('/kaggle/input/us-accidents-weekly-stage-2')

def resolve_artifact(filename, required=True):
    candidates = []
    if ARTIFACT_INPUT_ROOT is not None:
        p = Path(ARTIFACT_INPUT_ROOT) / filename
        if p.exists():
            candidates.append(p)
    p = ARTIFACT_DIR / filename
    if p.exists():
        candidates.append(p)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    # Deduplicate resolved paths and prefer the most recently modified attached artifact.
    unique = {str(p.resolve()): p for p in candidates}
    candidates = list(unique.values())
    if not candidates:
        if required:
            raise FileNotFoundError(
                f'Could not locate {filename}. Add the preceding notebook output as a Kaggle input, '
                f'or set ARTIFACT_INPUT_ROOT. Local output searched: {ARTIFACT_DIR}'
            )
        return None
    chosen = max(candidates, key=lambda p: p.stat().st_mtime)
    if len(candidates) > 1:
        print(f'Found {len(candidates)} copies of {filename}; using newest: {chosen}')
    return chosen

print('Project root:', PROJECT_ROOT.resolve())
print('Seed:', SEED)

import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_poisson_deviance,
)

PREDICTION_SOURCES = {
    'SGD — Squared Error': 'sgd_squared_test_predictions.csv',
    'SGD — Huber': 'sgd_huber_test_predictions.csv',
    'SGD — Epsilon-Insensitive': 'sgd_epsilon_insensitive_test_predictions.csv',
    'MLPRegressor': 'mlp_regressor_test_predictions.csv',
    'Incremental Mean Ensemble': 'incremental_ensemble_test_predictions.csv',
    'DCRNN': 'dcrnn_test_predictions.csv',
}
frames = {}
for name, filename in PREDICTION_SOURCES.items():
    path = resolve_artifact(filename)
    frame = pd.read_csv(path, parse_dates=['target_week_start'])
    frame = frame.sort_values('target_week_id').reset_index(drop=True)
    required = {'target_week_id','target_week_start','actual_count','predicted_count'}
    if not required.issubset(frame.columns):
        raise ValueError(f'{filename} lacks required columns {sorted(required-set(frame.columns))}')
    if frame.target_week_id.duplicated().any():
        raise AssertionError(f'Duplicate target week IDs in {filename}')
    frames[name] = frame
    print(name, '->', len(frame), 'test weeks')

reference = next(iter(frames.values()))
reference_ids = reference.target_week_id.to_numpy()
reference_actual = reference.actual_count.to_numpy(dtype=float)
for name, frame in frames.items():
    if not np.array_equal(frame.target_week_id.to_numpy(), reference_ids):
        raise AssertionError(f'{name} predictions do not cover the same target weeks.')
    if not np.allclose(frame.actual_count.to_numpy(dtype=float), reference_actual, rtol=0, atol=1e-5):
        raise AssertionError(f'{name} actual values differ from the reference actual series.')

# Exclude only weeks truncated by the global first/last source timestamps; this does not certify internal collection completeness.
national_path = resolve_artifact('weekly_national_counts.parquet')
national = pd.read_parquet(national_path)
exposure = national[['week_id','week_days_within_source_bounds']].rename(columns={'week_id':'target_week_id'})

## 1. Align forecasts and identify full seven-day test weeks

In [ ]:
comparison = reference[['target_week_id','target_week_start','actual_count']].copy()
for name, frame in frames.items():
    comparison[name] = frame.predicted_count.to_numpy(dtype=float)
comparison = comparison.merge(exposure, on='target_week_id', how='left', validate='one_to_one')
if comparison.week_days_within_source_bounds.isna().any():
    raise AssertionError('Could not recover weekly exposure metadata for one or more test predictions.')
comparison['is_boundary_complete_week'] = comparison.week_days_within_source_bounds == 7
full = comparison.loc[comparison.is_boundary_complete_week].reset_index(drop=True)
partial = comparison.loc[~comparison.is_boundary_complete_week].reset_index(drop=True)
if len(full) < 10:
    raise ValueError('Too few boundary-complete test weeks for reliable metrics.')
print('Boundary-complete test weeks:', len(full), '| truncated boundary weeks retained separately:', len(partial))
if len(partial):
    display(partial[['target_week_id','target_week_start','week_days_within_source_bounds','actual_count']])

## 2. Calculate the requested regression metrics

In [ ]:
# For adjusted R² we report a feature-count adjustment, not a full model-complexity adjustment.
feature_map_path = resolve_artifact('stream_i_feature_map.json')
stream_i_feature_count = len(json.loads(feature_map_path.read_text())['features'])
dcrnn_meta_path = resolve_artifact('dcrnn_metadata.json')
dcrnn_meta = json.loads(dcrnn_meta_path.read_text())
model_predictor_counts = {
    'SGD — Squared Error': stream_i_feature_count,
    'SGD — Huber': stream_i_feature_count,
    'SGD — Epsilon-Insensitive': stream_i_feature_count,
    'MLPRegressor': stream_i_feature_count,
    'Incremental Mean Ensemble': stream_i_feature_count,
    'DCRNN': int(dcrnn_meta.get('adjusted_r2_feature_count', stream_i_feature_count)),
}


def smape_percent(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    ratio = np.divide(2.0*np.abs(y_pred-y_true), denom,
                      out=np.zeros_like(denom, dtype=float), where=denom > 0)
    return float(100.0 * np.mean(ratio))


def adjusted_r2(r2, n, p):
    if n <= p + 1 or not np.isfinite(r2):
        return float('nan')
    return float(1.0 - (1.0-r2) * (n-1) / (n-p-1))


def regression_metrics(y_true, y_pred, p):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred_nonnegative = np.maximum(y_pred, 1e-9)  # Poisson deviance requires strictly positive predictions.
    r2 = float(r2_score(y_true, y_pred))
    denom = float(np.sum(np.abs(y_true)))
    wape = float(100.0*np.sum(np.abs(y_true-y_pred))/denom) if denom > 0 else float('nan')
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': r2,
        'Adjusted_R2_feature_count': adjusted_r2(r2, len(y_true), int(p)),
        'WAPE_percent': wape,
        'Poisson_Deviance': float(mean_poisson_deviance(y_true, y_pred_nonnegative)),
        'sMAPE_percent': smape_percent(y_true, y_pred),
    }

metric_rows=[]
for model_name in PREDICTION_SOURCES:
    row={'Model':model_name, 'n_test_weeks':len(full), 'p_used_for_adjusted_R2':model_predictor_counts[model_name]}
    row.update(regression_metrics(full.actual_count.to_numpy(), full[model_name].to_numpy(), model_predictor_counts[model_name]))
    metric_rows.append(row)
metrics = pd.DataFrame(metric_rows).sort_values('MAE').reset_index(drop=True)
display(metrics)
metrics.to_csv(REPORT_DIR/'final_regression_metrics.csv',index=False)

# Save unscored partial boundary observations alongside predictions for transparency.
if len(partial):
    partial.to_csv(REPORT_DIR/'partial_boundary_week_predictions.csv',index=False)

## 3. Naive forecasting references

These are reference baselines, not additional trained models. Persistence repeats the latest observed week's count. Seasonal naive uses the count from 52 weeks earlier. Both are evaluated only where their historical reference is available and the target week is complete.

In [ ]:
series = national.sort_values('week_id').set_index('week_id')['national_accident_count']
base = full[['target_week_id','target_week_start','actual_count']].copy()
base['Persistence'] = [float(series.loc[int(w)-1]) if int(w)-1 in series.index else np.nan for w in base.target_week_id]
base['Seasonal Naive (52w)'] = [float(series.loc[int(w)-52]) if int(w)-52 in series.index else np.nan for w in base.target_week_id]
for baseline_name in ['Persistence','Seasonal Naive (52w)']:
    valid = base[baseline_name].notna()
    if valid.sum() > 0:
        baseline_metrics = regression_metrics(base.loc[valid,'actual_count'],base.loc[valid,baseline_name],p=1)
        print(baseline_name, baseline_metrics)
        metrics = pd.concat([metrics, pd.DataFrame([{'Model':baseline_name,'n_test_weeks':int(valid.sum()),
            'p_used_for_adjusted_R2':1, **baseline_metrics}])],ignore_index=True)
metrics.to_csv(REPORT_DIR/'final_regression_metrics_with_baselines.csv',index=False)

## 4. Primary forecasting plot — actual series and all six models

The actual national count series is blue. Each model receives a separate line colour and a legend entry. Only complete seven-day target weeks are shown in the main benchmark plot.

In [ ]:
MODEL_COLORS = {
    'SGD — Squared Error':'#ff7f0e',
    'SGD — Huber':'#2ca02c',
    'SGD — Epsilon-Insensitive':'#d62728',
    'MLPRegressor':'#9467bd',
    'Incremental Mean Ensemble':'#8c564b',
    'DCRNN':'#e377c2',
}
fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(full.target_week_start, full.actual_count, color='blue', linewidth=2.2, label='Actual', zorder=5)
for name in PREDICTION_SOURCES:
    ax.plot(full.target_week_start, full[name], color=MODEL_COLORS[name], linewidth=1.35, alpha=0.9, label=name)
ax.set_title('Weekly US accident-count forecasting — actual vs six models')
ax.set_xlabel('Target week (Monday start)'); ax.set_ylabel('Accident reports')
ax.grid(alpha=0.25); ax.legend(loc='upper left', bbox_to_anchor=(1.01,1.0), borderaxespad=0.)
fig.tight_layout()
fig.savefig(REPORT_DIR/'01_actual_vs_all_model_forecasts.png',dpi=180,bbox_inches='tight')
plt.show()

## 5. Actual-versus-predicted scatter plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, name in zip(axes.ravel(), PREDICTION_SOURCES):
    actual = full.actual_count.to_numpy(dtype=float)
    predicted = full[name].to_numpy(dtype=float)
    low = min(float(actual.min()), float(predicted.min()))
    high = max(float(actual.max()), float(predicted.max()))
    ax.scatter(actual, predicted, s=22, alpha=0.65, color=MODEL_COLORS[name])
    ax.plot([low,high],[low,high],linestyle='--',color='black',linewidth=1)
    ax.set_title(name); ax.set_xlabel('Actual count'); ax.set_ylabel('Predicted count')
    ax.grid(alpha=0.2)
fig.suptitle('Actual vs predicted weekly accident counts',y=1.01,fontsize=15)
fig.tight_layout()
fig.savefig(REPORT_DIR/'02_actual_vs_predicted_scatter.png',dpi=180,bbox_inches='tight')
plt.show()

## 6. Residuals over time and residual distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True)
for ax, name in zip(axes.ravel(), PREDICTION_SOURCES):
    residual = full[name].to_numpy(dtype=float) - full.actual_count.to_numpy(dtype=float)
    ax.axhline(0,color='black',linewidth=1,linestyle='--')
    ax.plot(full.target_week_start,residual,color=MODEL_COLORS[name],linewidth=1.1)
    ax.set_title(name); ax.set_ylabel('Prediction − actual')
    ax.grid(alpha=0.2)
fig.suptitle('Forecast residuals over time',y=1.01,fontsize=15)
fig.tight_layout()
fig.savefig(REPORT_DIR/'03_residuals_over_time.png',dpi=180,bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(14,5))
for name in PREDICTION_SOURCES:
    residual = full[name].to_numpy(dtype=float)-full.actual_count.to_numpy(dtype=float)
    ax.hist(residual,bins=24,alpha=0.28,label=name,color=MODEL_COLORS[name])
ax.set_title('Residual distributions by model')
ax.set_xlabel('Prediction − actual'); ax.set_ylabel('Number of weeks')
ax.legend(ncol=2); ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(REPORT_DIR/'04_residual_distributions.png',dpi=180,bbox_inches='tight')
plt.show()

## 7. Error comparison and rolling 13-week MAE

In [ ]:
core_metrics = metrics[metrics.Model.isin(PREDICTION_SOURCES)].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(core_metrics.Model,core_metrics.MAE)
axes[0].set_title('MAE by model'); axes[0].set_ylabel('Accident reports')
axes[0].tick_params(axis='x',rotation=35); axes[0].grid(axis='y',alpha=0.2)
axes[1].bar(core_metrics.Model,core_metrics.WAPE_percent)
axes[1].set_title('WAPE by model'); axes[1].set_ylabel('Percent')
axes[1].tick_params(axis='x',rotation=35); axes[1].grid(axis='y',alpha=0.2)
fig.tight_layout()
fig.savefig(REPORT_DIR/'05_error_metric_comparison.png',dpi=180,bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(17,5))
for name in PREDICTION_SOURCES:
    abs_err = np.abs(full[name].to_numpy(dtype=float)-full.actual_count.to_numpy(dtype=float))
    rolling = pd.Series(abs_err).rolling(13,min_periods=4).mean()
    ax.plot(full.target_week_start,rolling,label=name,color=MODEL_COLORS[name])
ax.set_title('Rolling 13-week mean absolute error')
ax.set_xlabel('Target week'); ax.set_ylabel('Rolling MAE')
ax.legend(ncol=2); ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(REPORT_DIR/'06_rolling_13_week_mae.png',dpi=180,bbox_inches='tight')
plt.show()

## 8. Save the consolidated benchmark table and report summary

Adjusted R² here uses the number of engineered input columns for Stream I and the number of DCRNN input/covariate channels for DCRNN. This is only a feature-count adjustment. It does not account for MLP hidden units, DCRNN recurrent parameters, graph operations, or ensemble complexity, so interpret it as a supplementary statistic rather than a fair complexity correction across all architectures.

Poisson deviance requires strictly positive predicted means; predictions are clipped to a small positive value **for this metric only**. Reported model predictions remain unrounded and non-negative in their saved files.

In [ ]:
# Save one merged wide prediction table for reproducibility.
wide = full[['target_week_id','target_week_start','week_days_within_source_bounds','actual_count'] + list(PREDICTION_SOURCES)].copy()
wide.to_csv(REPORT_DIR/'all_models_weekly_test_predictions_wide.csv',index=False)

summary = {
    'target':'national weekly accident report count',
    'frequency':'weekly', 'horizon':'one week ahead',
    'test_protocol':'frozen fitted model weights; rolling-origin one-step forecasts with observed history through each forecast origin',
    'n_boundary_complete_test_weeks':int(len(full)),
    'n_partial_boundary_weeks':int(len(partial)),
    'models':list(PREDICTION_SOURCES),
    'metrics':['MAE','RMSE','R2','Adjusted_R2_feature_count','WAPE_percent','Poisson_Deviance','sMAPE_percent'],
    'log_loss':'Not used: classification probability loss; Poisson deviance is used for count forecasts.',
    'adjusted_r2_caveat':'Feature-count adjusted R2 is not a full complexity correction and is not directly comparable for nonlinear/neural/ensemble models.',
    'primary_plot':'01_actual_vs_all_model_forecasts.png',
    'best_by_mae':str(metrics[metrics.Model.isin(PREDICTION_SOURCES)].sort_values('MAE').iloc[0]['Model']),
}
(REPORT_DIR/'final_benchmark_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('Final reports and plots saved to:',REPORT_DIR)